In [ ]:
# Phase 5 — Daily Narrative Generator
# Reads watchlist, predictions, and anomaly data to produce
# a human-readable daily briefing for Power BI display.
# Output: ml.daily_narrative (Delta table)
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import json
LAKEHOUSE = "PI-Gold-Delta"
SCHEMA    = "ml"
# Central Time offset (CDT = UTC-5)
CT_OFFSET_HOURS = 5
now_utc = datetime.utcnow()
now_ct  = now_utc - timedelta(hours=CT_OFFSET_HOURS)
today_str = now_ct.strftime("%Y-%m-%d")
time_str  = now_ct.strftime("%I:%M %p CT")
print(f"Generating narrative for {today_str} at {time_str}")

In [ ]:
# ── Read current data (latest run per model) + previous runs ─
watchlist = spark.sql(f"""
    SELECT w.* FROM {SCHEMA}.watchlist w
    INNER JOIN (
        SELECT model_name, scoring_date, MAX(model_run_timestamp) AS latest_ts
        FROM {SCHEMA}.watchlist
        WHERE scoring_date = (SELECT MAX(scoring_date) FROM {SCHEMA}.watchlist)
        GROUP BY model_name, scoring_date
    ) latest ON w.model_name = latest.model_name
              AND w.scoring_date = latest.scoring_date
""")
# Previous day's watchlist for day-over-day comparison
watchlist_prev = spark.sql(f"""
    SELECT w.* FROM {SCHEMA}.watchlist w
    INNER JOIN (
        SELECT model_name, scoring_date, MAX(model_run_timestamp) AS latest_ts
        FROM {SCHEMA}.watchlist
        WHERE scoring_date = (
            SELECT MAX(scoring_date) FROM {SCHEMA}.watchlist
            WHERE scoring_date < (SELECT MAX(scoring_date) FROM {SCHEMA}.watchlist)
        )
        GROUP BY model_name, scoring_date
    ) prev ON w.model_name = prev.model_name
            AND w.scoring_date = prev.scoring_date
""")
pred_short = spark.sql(f"""
    SELECT p.* FROM {SCHEMA}.predictions_shortterm p
    INNER JOIN (
        SELECT asset_id, prediction_horizon, label_type, MAX(scoring_timestamp) AS max_ts
        FROM {SCHEMA}.predictions_shortterm
        GROUP BY asset_id, prediction_horizon, label_type
    ) latest ON p.asset_id = latest.asset_id
            AND p.prediction_horizon = latest.prediction_horizon
            AND p.label_type = latest.label_type
            AND p.scoring_timestamp = latest.max_ts
""")
pred_long = spark.sql(f"""
    SELECT p.* FROM {SCHEMA}.predictions_longterm p
    INNER JOIN (
        SELECT asset_id, MAX(scoring_timestamp) AS max_ts
        FROM {SCHEMA}.predictions_longterm
        GROUP BY asset_id
    ) latest ON p.asset_id = latest.asset_id
            AND p.scoring_timestamp = latest.max_ts
""")
# Previous predictions for trend comparison
pred_short_prev = spark.sql(f"""
    SELECT p.* FROM {SCHEMA}.predictions_shortterm p
    INNER JOIN (
        SELECT asset_id, prediction_horizon, label_type, MAX(scoring_timestamp) AS max_ts
        FROM {SCHEMA}.predictions_shortterm
        WHERE scoring_timestamp < (SELECT MAX(scoring_timestamp) FROM {SCHEMA}.predictions_shortterm)
        GROUP BY asset_id, prediction_horizon, label_type
    ) prev ON p.asset_id = prev.asset_id
           AND p.prediction_horizon = prev.prediction_horizon
           AND p.label_type = prev.label_type
           AND p.scoring_timestamp = prev.max_ts
""")
pred_long_prev = spark.sql(f"""
    SELECT p.* FROM {SCHEMA}.predictions_longterm p
    INNER JOIN (
        SELECT asset_id, MAX(scoring_timestamp) AS max_ts
        FROM {SCHEMA}.predictions_longterm
        WHERE scoring_timestamp < (SELECT MAX(scoring_timestamp) FROM {SCHEMA}.predictions_longterm)
        GROUP BY asset_id
    ) prev ON p.asset_id = prev.asset_id
           AND p.scoring_timestamp = prev.max_ts
""")
# Bridge table for friendly tag names
bridge = spark.sql("SELECT Tag, tag_description FROM gold.bridge_pi_tag_to_asset")
bridge_map = {r["Tag"]: r["tag_description"] for r in bridge.collect()}
# Collect to Python dicts
wl_rows = [r.asDict() for r in watchlist.collect()]
wl_prev = [r.asDict() for r in watchlist_prev.collect()]
ps_rows = [r.asDict() for r in pred_short.collect()]
pl_rows = [r.asDict() for r in pred_long.collect()]
ps_prev = [r.asDict() for r in pred_short_prev.collect()]
pl_prev = [r.asDict() for r in pred_long_prev.collect()]
# Build previous-day lookup: (model_name, asset_id, tag_name) -> row
prev_lookup = {}
for r in wl_prev:
    key = (r.get("model_name",""), r.get("asset_id",""), r.get("tag_name",""))
    prev_lookup[key] = r
print(f"Watchlist entries (latest scoring_date per model): {len(wl_rows)}")
for m in set(r["model_name"] for r in wl_rows):
    ts = max(r["model_run_timestamp"] for r in wl_rows if r["model_name"] == m)
    cnt = sum(1 for r in wl_rows if r["model_name"] == m)
    print(f"  {m}: {ts} ({cnt} rows)")
print(f"Previous watchlist: {len(wl_prev)} rows")
print(f"Short-term predictions: {len(ps_rows)} current, {len(ps_prev)} previous")
print(f"Long-term predictions:  {len(pl_rows)} current, {len(pl_prev)} previous")

In [ ]:
# ── Narrative Generation (Priority Actions + Trends) ─────────
TARGET_ASSETS = ["RV2_U2_Boiler", "RV3_U3_Steam_Turbine", "RV3_U3_Boiler_Feed_Pump_East"]
ASSET_FRIENDLY = {
    "RV2_U2_Boiler":              "RV2 Unit 2 Boiler",
    "RV3_U3_Steam_Turbine":       "RV3 Unit 3 Steam Turbine",
    "RV3_U3_Boiler_Feed_Pump_East": "RV3 Boiler Feed Pump East",
}
# ── Organize data by asset ──
asset_data = {}
for a in TARGET_ASSETS:
    asset_data[a] = {
        "watchlist": [r for r in wl_rows if r["asset_id"] == a],
        "short":     [r for r in ps_rows if r["asset_id"] == a],
        "long":      [r for r in pl_rows if r["asset_id"] == a],
        "short_prev": [r for r in ps_prev if r["asset_id"] == a],
        "long_prev":  [r for r in pl_prev if r["asset_id"] == a],
    }
# ── Count severities ──
critical_count = sum(1 for r in wl_rows if r["recommended_action"] == "CRITICAL")
medium_count   = sum(1 for r in wl_rows if r["recommended_action"] == "MEDIUM")
total_alerts   = len(wl_rows)
assets_flagged = len(set(r["asset_id"] for r in wl_rows if r["recommended_action"] in ("CRITICAL","HIGH","MEDIUM")))
if critical_count > 0:
    sys_status_text = "CRITICAL"
elif medium_count > 0:
    sys_status_text = "MODERATE"
else:
    sys_status_text = "HEALTHY"
# ── Row-specific trend with day-over-day comparison ──
def row_trend_html(w):
    """Build trend indicator with day-over-day comparison."""
    direction = (w.get("trend_direction") or "").upper()
    slope     = w.get("trend_slope_per_day")
    current   = w.get("current_value")
    baseline  = w.get("baseline_mean")
    units     = w.get("engineering_units") or ""
    parts = []
    if direction == "RISING" and slope is not None:
        parts.append(f'<span style="color:#dc3545;font-size:10px">&#9650; +{abs(slope):.2f}/{units}/d</span>')
    elif direction == "FALLING" and slope is not None:
        parts.append(f'<span style="color:#28a745;font-size:10px">&#9660; {abs(slope):.2f}/{units}/d</span>')
    elif direction == "STABLE":
        parts.append('<span style="color:#999;font-size:10px">&#8596; stable</span>')
    if current is not None and baseline is not None and baseline != 0:
        dev_pct = ((current - baseline) / abs(baseline)) * 100
        if abs(dev_pct) >= 1:
            dev_color = "#dc3545" if dev_pct > 0 else "#28a745"
            parts.append(f'<span style="color:{dev_color};font-size:10px">{dev_pct:+.0f}% vs 30d avg</span>')
    key = (w.get("model_name",""), w.get("asset_id",""), w.get("tag_name",""))
    prev = prev_lookup.get(key)
    if prev and current is not None:
        prev_val = prev.get("current_value")
        prev_risk = prev.get("risk_contribution")
        curr_risk = w.get("risk_contribution")
        if prev_val is not None and prev_val != 0:
            dod_pct = ((current - prev_val) / abs(prev_val)) * 100
            if abs(dod_pct) >= 1:
                if dod_pct > 0:
                    dod_icon = "&#9650;"
                    dod_word = "worse" if w.get("recommended_action") in ("CRITICAL","HIGH") else "up"
                else:
                    dod_icon = "&#9660;"
                    dod_word = "improved" if w.get("recommended_action") in ("CRITICAL","HIGH") else "down"
                dod_color = "#dc3545" if dod_word in ("worse","up") else "#28a745"
                parts.append(f'<span style="color:{dod_color};font-size:10px">{dod_icon} {abs(dod_pct):.0f}% {dod_word} vs yesterday</span>')
            else:
                parts.append('<span style="color:#999;font-size:10px">&#8596; unchanged vs yesterday</span>')
        if curr_risk is not None and prev_risk is not None:
            risk_delta = curr_risk - prev_risk
            if abs(risk_delta) >= 1:
                risk_color = "#dc3545" if risk_delta > 0 else "#28a745"
                risk_word = "risk up" if risk_delta > 0 else "risk down"
                parts.append(f'<span style="color:{risk_color};font-size:10px">{risk_word} {abs(risk_delta):.1f} ({prev_risk:.1f}&rarr;{curr_risk:.1f})</span>')
    return " &nbsp;|&nbsp; ".join(parts) if parts else ""

def consolidate_by_asset(wl_entries, level):
    """Merge multiple watchlist entries of the same level for one asset into a single action."""
    all_tags = []
    worst_z = 0
    worst_row = None
    sources = set()
    for w in wl_entries:
        text = w.get("recommendation_text", "")
        # Extract tag names from "Watch Tag1, Tag2, Tag3. Worst anomaly z=X."
        import re
        tag_match = re.match(r'Watch (.+?)\.', text)
        if tag_match:
            tags = [t.strip() for t in tag_match.group(1).split(",")]
            all_tags.extend(tags)
        z_match = re.search(r'z=([\d]+\.?\d*)', text)
        z_val = float(z_match.group(1)) if z_match else 0
        if z_val > worst_z:
            worst_z = z_val
            worst_row = w
        sources.add(w.get("model_name", ""))
    # Dedupe tags, keep order, limit to top 5
    seen = set()
    unique_tags = []
    for t in all_tags:
        if t not in seen:
            seen.add(t)
            unique_tags.append(t)
    top_tags = unique_tags[:5]
    tag_list = ", ".join(top_tags)
    extra = f" (+{len(unique_tags) - 5} more)" if len(unique_tags) > 5 else ""
    src_str = "+".join(sorted(sources))
    merged_text = f"Watch {tag_list}{extra}. Worst anomaly z={worst_z:.1f}. Sources: {src_str}."
    return merged_text, worst_row

# ── Build priority actions list (consolidated per asset per level) ──
actions = []
for level in ["CRITICAL", "HIGH"]:
    for asset_id in TARGET_ASSETS:
        wl = asset_data[asset_id]["watchlist"]
        friendly = ASSET_FRIENDLY.get(asset_id, asset_id)
        level_entries = [w for w in wl if w["recommended_action"] == level]
        if level_entries:
            merged_text, worst_row = consolidate_by_asset(level_entries, level)
            actions.append((level, friendly, merged_text, asset_id, worst_row))

for asset_id in TARGET_ASSETS:
    friendly = ASSET_FRIENDLY.get(asset_id, asset_id)
    short = asset_data[asset_id]["short"]
    high_prob = [r for r in short if r["prediction_horizon"] == "4h" and r["stop_probability"] >= 0.4]
    if high_prob:
        prob = high_prob[0]["stop_probability"]
        prev_short = asset_data[asset_id]["short_prev"]
        prev_4h = [r for r in prev_short if r["prediction_horizon"] == "4h"]
        if prev_4h:
            prev_prob = prev_4h[0]["stop_probability"]
            chg = prob - prev_prob
            chg_str = f" ({chg:+.0%} vs yesterday)" if abs(chg) >= 0.01 else " (unchanged)"
        else:
            chg_str = ""
        actions.append(("MONITOR", friendly, f"{prob:.0%} 4h stop probability{chg_str}", asset_id, None))

for asset_id in TARGET_ASSETS:
    wl = asset_data[asset_id]["watchlist"]
    friendly = ASSET_FRIENDLY.get(asset_id, asset_id)
    medium_entries = [w for w in wl if w["recommended_action"] == "MEDIUM"]
    if medium_entries:
        merged_text, worst_row = consolidate_by_asset(medium_entries, "MEDIUM")
        actions.append(("MEDIUM", friendly, merged_text, asset_id, worst_row))

# ── Asset-level day-over-day summary ──
asset_dod_html = []
for asset_id in TARGET_ASSETS:
    friendly = ASSET_FRIENDLY.get(asset_id, asset_id)
    curr_long = [r for r in pl_rows if r["asset_id"] == asset_id]
    prev_long = [r for r in pl_prev if r["asset_id"] == asset_id]
    if curr_long and prev_long:
        c14 = curr_long[0]["survival_probability_14d"]
        p14 = prev_long[0]["survival_probability_14d"]
        chg = (c14 - p14) * 100
        if abs(chg) >= 0.1:
            icon = "&#9650;" if chg > 0 else "&#9660;"
            color = "#28a745" if chg > 0 else "#dc3545"
            word = "improved" if chg > 0 else "declined"
            asset_dod_html.append(f'<span style="color:{color};font-size:10px">{icon} {friendly}: 14d survival {word} {abs(chg):.1f}pp ({p14*100:.1f}% &rarr; {c14*100:.1f}%)</span>')
        else:
            asset_dod_html.append(f'<span style="color:#999;font-size:10px">&#8596; {friendly}: 14d survival stable at {c14*100:.1f}%</span>')
# ── Build HTML (compact table with per-row trends) ──
h = []
h.append('<table style="font-family:\'Segoe UI\',Arial,sans-serif;width:100%;border-collapse:collapse;font-size:11px;line-height:1.3">')
if asset_dod_html:
    h.append(f'<tr><td colspan="4" style="padding:4px;border-bottom:2px solid #ccc;background:#f8f9fa">')
    h.append(f'<strong style="font-size:11px">Day-over-day:</strong> &nbsp; ' + " &nbsp;|&nbsp; ".join(asset_dod_html))
    h.append(f'</td></tr>')
if actions:
    for i, (level, asset, text, asset_id, wl_row) in enumerate(actions, 1):
        colors = {"CRITICAL": "#dc3545", "HIGH": "#e8a317", "MEDIUM": "#e8a317", "MONITOR": "#0d6efd"}
        lc = colors.get(level, "#6c757d")
        trend_html = row_trend_html(wl_row) if wl_row else ""
        trend_cell = f'<td style="padding:2px 4px;border-bottom:1px solid #e8e8e8;white-space:nowrap;text-align:right">{trend_html}</td>' if trend_html else ''
        h.append(f'<tr>')
        h.append(f'<td style="padding:2px 4px;white-space:nowrap;color:{lc};font-weight:700;vertical-align:top;border-bottom:1px solid #e8e8e8">[{level}]</td>')
        h.append(f'<td style="padding:2px 4px;font-weight:600;white-space:nowrap;vertical-align:top;border-bottom:1px solid #e8e8e8">{asset}</td>')
        h.append(f'<td style="padding:2px 4px;border-bottom:1px solid #e8e8e8">{text}</td>')
        h.append(trend_cell)
        h.append(f'</tr>')
else:
    h.append('<tr><td style="padding:2px 4px;color:#28a745;font-weight:600" colspan="4">&#10004; No priority actions. All systems nominal.</td></tr>')
h.append('</table>')
narrative_html = "\n".join(h)
narrative_text = " | ".join(f"[{level}] {asset}: {text}" for level, asset, text, _, _ in actions) if actions else "No priority actions."
print("HTML narrative generated:", len(narrative_html), "chars")
print(f"Actions: {len(actions)}")
for level, asset, text, aid, wl_row in actions:
    print(f"  [{level}] {asset}: {text[:80]}")

In [ ]:
# ── Write narrative to Delta table ────────────────────────────
import pandas as pd
narrative_df = pd.DataFrame([{
    "narrative_date":   today_str,
    "narrative_text":   narrative_text,
    "narrative_html":   narrative_html,
    "system_status":    sys_status_text,
    "critical_alerts":  critical_count,
    "total_alerts":     total_alerts,
    "assets_flagged":   assets_flagged,
    "generated_at":     now_utc.strftime("%Y-%m-%dT%H:%M:%S"),
}])
schema = StructType([
    StructField("narrative_date",  StringType(),  True),
    StructField("narrative_text",  StringType(),  True),
    StructField("narrative_html",  StringType(),  True),
    StructField("system_status",   StringType(),  True),
    StructField("critical_alerts", IntegerType(), True),
    StructField("total_alerts",    IntegerType(), True),
    StructField("assets_flagged",  IntegerType(), True),
    StructField("generated_at",    StringType(),  True),
])
sdf = spark.createDataFrame(narrative_df, schema=schema)
sdf.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA}.daily_narrative")
print(f"\u2713 Narrative written to {SCHEMA}.daily_narrative for {today_str}")
print(f"  HTML: {len(narrative_html)} chars")
print(f"  Text: {len(narrative_text)} chars")